# Auto Post to Facebook — Kaggle Free GPU

Run one full cycle of the **Auto Posting Facebook** project on Kaggle's **free GPU** (P100 or T4):

**News** → **Caption + image prompt** → **Image (Z-Image-Turbo on GPU)** → **Overlay** → **Post to Facebook**

**Caption & image prompt:** Locally the project can use **Ollama** (see `config.py` → `USE_OLLAMA`). On Kaggle we use **Gemini** because Ollama runs only on your machine and isn't available in the cloud.

---

## Setup (do once)

1. **Enable GPU**  
   - Right panel → **Settings** (gear icon) → **Accelerator** → select **GPU** (P100 or T4 x2).  
   - Save. The session will use Kaggle's free GPU quota (~30 hrs/week).

2. **Add Secrets**  
   - Right panel → **Add-ons** → **Secrets**.  
   - Create and attach these (name = label, value = your key):  
     - `GEMINI_API_KEY` — [Google AI Studio](https://aistudio.google.com/apikey) API key  
     - `NEWS_API_KEY` — [newsdata.io](https://newsdata.io/) key *(optional; if missing, uses BBC headline)*  
     - `PAGE_ACCESS_TOKEN` — Facebook Page access token  
     - `PAGE_ID` — Facebook Page ID  

3. **Run**  
   - **Run** → **Run All**.  
   - First run downloads the model (~2–4 min); then image generation runs on GPU.  
   - Output image: `/kaggle/working/post_image.jpg` (download from Output tab or use **Add-ons** → **Output**).

---

Based on [Auto Posting Facebook](https://github.com/...) — run the same pipeline locally or on Kaggle.

In [ ]:
# Cell 1: Load secrets and check GPU
import os, sys, subprocess

try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for key in ["GEMINI_API_KEY", "NEWS_API_KEY", "PAGE_ACCESS_TOKEN", "PAGE_ID"]:
        try:
            os.environ[key] = s.get_secret(key)
        except Exception:
            pass
except Exception as e:
    print("Secrets:", e)

# Gemini: some environments expect GOOGLE_API_KEY
if os.environ.get("GEMINI_API_KEY") and not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai", "python-dotenv", "feedparser"], capture_output=True)

import torch
gpu_ok = torch.cuda.is_available()
print("GPU:", gpu_ok, torch.cuda.get_device_name(0) if gpu_ok else "(using CPU — enable GPU in Settings)")
if not gpu_ok:
    print("Tip: Settings → Accelerator → GPU, then re-run.")

In [ ]:
# Cell 2: Fetch one news article (Newsdata.io or BBC RSS fallback)
import requests

article = None
api_key = os.environ.get("NEWS_API_KEY", "").strip()

if api_key:
    try:
        r = requests.get("https://newsdata.io/api/1/latest", params={"apikey": api_key, "language": "en", "country": "us"}, timeout=15)
        r.raise_for_status()
        res = r.json().get("results", [])
        if res:
            a = res[0]
            article = {
                "title": (a.get("title") or a.get("title_full", ""))[:200],
                "description": (a.get("description") or a.get("content") or "")[:500],
                "url": a.get("link") or a.get("url", ""),
                "source": a.get("source_id", "News"),
            }
            print("Article (newsdata.io):", article["title"][:60], "...")
    except Exception as e:
        print("Newsdata.io error:", e)

if not article:
    try:
        import feedparser
        fp = feedparser.parse("https://feeds.bbci.co.uk/news/rss.xml", request_headers={"User-Agent": "KaggleNotebook/1.0"})
        entries = fp.get("entries", [])
        if entries:
            e = entries[0]
            article = {
                "title": (e.get("title") or "")[:200],
                "description": (e.get("summary", "") or e.get("title", ""))[:500],
                "url": e.get("link", ""),
                "source": "BBC News",
            }
            print("Article (BBC RSS):", article["title"][:60], "...")
    except Exception as e:
        print("BBC RSS error:", e)

if not article:
    article = {"title": "Markets Open Higher Amid Earnings Wave", "description": "Stocks rise as investors digest quarterly results.", "url": "", "source": "Sample"}
    print("Using sample article. Add NEWS_API_KEY in Secrets for newsdata.io, or rely on BBC RSS.")

In [ ]:
# Cell 3: Gemini caption + image prompt (google-genai SDK only)
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY", "").strip()
if not api_key:
    raise ValueError("Set GEMINI_API_KEY in Kaggle Secrets (Add-ons → Secrets). Get a key at https://aistudio.google.com/apikey")

from google import genai
client = genai.Client(api_key=api_key)

# Model IDs that work with current Gemini API (try in order)
GEMINI_MODELS = ["gemini-2.0-flash", "gemini-1.5-flash-latest", "gemini-1.5-pro-latest", "gemini-1.5-pro", "gemini-pro"]
gemini_model = None
for m in GEMINI_MODELS:
    try:
        r = client.models.generate_content(model=m, contents="Say OK")
        if getattr(r, "text", None) or (getattr(r, "candidates", None) and len(r.candidates) > 0):
            gemini_model = m
            print("Using model:", gemini_model)
            break
    except Exception:
        continue
if not gemini_model:
    gemini_model = "gemini-2.0-flash"
    print("Using default model:", gemini_model)

def _gemini_text(r):
    if hasattr(r, "text") and r.text:
        return r.text
    if hasattr(r, "candidates") and r.candidates:
        c = r.candidates[0]
        if hasattr(c, "content") and c.content and hasattr(c.content, "parts"):
            for part in c.content.parts:
                if hasattr(part, "text") and part.text:
                    return part.text
    return ""

def gemini_caption(art):
    p = f"""You are a senior US news editor. Based on this article write a Facebook caption.
Title: {art['title']}
Description: {art['description']}
Start with BREAKING: or JUST IN:. 180-250 words. End with 20-30 hashtags. No emojis. Return only the caption."""
    r = client.models.generate_content(model=gemini_model, contents=p)
    return (_gemini_text(r) or "").strip()

def gemini_image_prompt(art):
    p = f"""One sentence image prompt for photorealistic, dramatic news-style image about: {art['title']}. No text in image. Vivid."""
    r = client.models.generate_content(model=gemini_model, contents=p)
    return ((_gemini_text(r) or "").strip().strip('"'))[:500]

caption = gemini_caption(article)
image_prompt = gemini_image_prompt(article)
print("Caption len:", len(caption), "| Prompt:", image_prompt[:70], "...")

In [ ]:
# Cell 4: Z-Image-Turbo on Kaggle GPU
from diffusers import DiffusionPipeline
import torch, gc

try:
    from diffusers.utils import logging as diffusers_logging
    diffusers_logging.disable_progress_bar()
except Exception:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = os.environ.get("Z_IMAGE_TURBO_MODEL", "aiorbust/z-image-turbo")
print("Loading", model_id, "on", device, "...")

pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
if device == "cuda":
    pipe.enable_model_cpu_offload()
else:
    pipe = pipe.to(device)

prompt = (image_prompt or "Professional news scene") + " Vivid, saturated colors."
out_path = "/kaggle/working/post_image.jpg"

image = pipe(prompt=prompt, height=1280, width=1024, num_inference_steps=8, guidance_scale=0.0).images[0]
image.save(out_path)
print("Saved:", out_path)

del pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Cell 5: Headline overlay
from PIL import Image, ImageDraw, ImageFont

img = Image.open(out_path).convert("RGB")
draw = ImageDraw.Draw(img)
w, h = img.size
bar_h = int(h * 0.25)
draw.rectangle([0, h - bar_h, w, h], fill=(20, 30, 60))

headline = " ".join((article.get("title") or "Breaking News")[:80].split()[:10])
try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", min(48, w // 18))
except Exception:
    font = ImageFont.load_default()

bbox = draw.textbbox((0, 0), headline, font=font)
tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
draw.text(((w - tw) // 2, h - bar_h + (bar_h - th) // 2), headline, fill=(255, 255, 255), font=font)
img.save(out_path, quality=95)
print("Overlay done.")

In [ ]:
# Cell 6: Post to Facebook
pid = os.environ.get("PAGE_ID", "").strip()
token = os.environ.get("PAGE_ACCESS_TOKEN", "").strip()

if not pid or not token:
    print("Add PAGE_ID and PAGE_ACCESS_TOKEN in Secrets to post to Facebook.")
    print("Image saved to /kaggle/working/post_image.jpg — download from Output tab.")
else:
    url = f"https://graph.facebook.com/v21.0/{pid}/photos"
    with open(out_path, "rb") as f:
        r = requests.post(url, files={"source": f}, data={"access_token": token, "caption": caption[:63000], "published": "true"}, timeout=60)
    if r.ok:
        print("SUCCESS: Posted to Facebook.")
    else:
        print(f"Post failed: {r.status_code}", r.text[:300])

In [ ]:
# Cell 7: Display generated image
from PIL import Image
display(Image.open(out_path))